# GCM BA - 0. Precipitation regimes: quick check of the raw data before choosing the BA method

No bias adjustment here: only the raw inputs (ERA5 on the EC-Earth3 grid, EC-Earth3 historical, the four GCM runs),
over a region (Europe by default, `REGION = None` for the whole globe, much slower), on the LISFLOOD domain. Three
questions that decide which choices of the F1 method matter (see `GCM_BA_F1_method_notes.docx`):

1. **Dry days and drizzle**: fraction of days < 0.1 mm and < 1 mm per month, ERA5 vs EC-Earth3 (historical and runs):
   how much work the dry-day treatment (fixed WDF or jitter) has to do.
2. **Seasonal cycle of the bias** (ERA5 - EC-Earth3 historical, monthly means, pr and tasmax): how much it changes from
   one month to the next (source of the jumps of a monthly correction), and with a 3-month smoothing (Cannon et al. 2015).
3. **Extremes beyond the historical range**: per pixel and month, hq = 99.5th percentile of EC-Earth3 historical (all
   days, 3-month window as in the new F1); share of the runs' wet days above hq and how far above it (x / hq). If few
   days exceed hq, or only slightly, the choice of the rule above hq is secondary.

Nothing is written except the figures in `outputs/` next to this notebook.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
cfg = bu.load_config(CONFIG_PATH)
REGION = {"lat": (35, 72), "lon": (-25, 45)}   # Europe | None = whole globe (much slower)
CAL_PERIOD = ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
LABEL = {"era5": "ERA5", "gcm_historical": "EC-Earth3 historical", "gcm_2025_nohosing": "2025 original",
         "gcm_2025_hosing": "2025 hosing", "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
Q_TOP = 0.995                          # last quantile of the new F1 (100 quantiles on all days)
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)


def region(da):
    return da if REGION is None else da.sel(lat=slice(*REGION["lat"]), lon=slice(*REGION["lon"]))


def load(name, var):
    """Raw input on the EC-Earth3 grid, cropped to REGION and loaded (historical / ERA5 over CAL_PERIOD)."""
    period = CAL_PERIOD if name in ("era5", "gcm_historical") else None
    return region(bu.gcm_ba_input(cfg, name, var, period)).load()


MASK = region(bu.load_domain_mask(cfg, "ecearth3")).values.astype(bool)            # LISFLOOD domain (land)
W = np.cos(np.deg2rad(region(bu.load_domain_mask(cfg, "ecearth3")).lat.values))[:, None] * MASK


def rmean(a):
    """Area-weighted mean over the land pixels of the region of a (..., lat, lon) array."""
    a = np.where(MASK, a, np.nan)
    return np.nansum(a * W, axis=(-2, -1)) / np.sum(W * np.isfinite(a), axis=(-2, -1))


PR = {n: load(n, "pr") for n in ["era5", "gcm_historical"] + RUNS if bu.gcm_ba_available(cfg, n, "pr")}
print({LABEL[n]: dict(d.sizes) for n, d in PR.items()})

## 1. Dry days and drizzle (fraction of days, per month, land mean)

In [ ]:
rows = []
for n, d in PR.items():
    mon = d.time.dt.month.values
    for thr in (0.1, 1.0):
        frac = [rmean((d.values[mon == m] < thr).mean(0)) for m in range(1, 13)]
        rows.append({"dataset": LABEL[n], "threshold (mm/day)": thr, **{str(m): f for m, f in zip(range(1, 13), frac)},
                     "year": float(np.mean(frac))})
dry = pd.DataFrame(rows).set_index(["threshold (mm/day)", "dataset"]).round(3)
display(dry)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, thr in zip(axes, (0.1, 1.0)):
    for n in PR:
        ax.plot(range(1, 13), dry.loc[(thr, LABEL[n])].values[:12], "o-", ms=3, label=LABEL[n],
                lw=2.5 if n == "era5" else 1.2, color="k" if n == "era5" else None)
    ax.set_title(f"fraction of days < {thr} mm/day (land, region)", fontsize=9); ax.set_xlabel("month")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "dry_days.png", dpi=110); plt.show()
print("Gap between the 0.1 and 1 mm rows = drizzle days. ERA5 vs historical = what the dry-day treatment must correct.")

## 2. Seasonal cycle of the bias and jumps between months

Bias = ERA5 - EC-Earth3 historical, monthly climatology per pixel (pr in %, tasmax in degC). A monthly correction
follows this bias month by month, so the change of the bias between two consecutive months (per pixel) is roughly the
jump of the correction at that change of month. "3-month" = the same with each month's bias averaged over the 3 months
centred on it (as with the Cannon et al. 2015 calibration windows).

In [ ]:
def monthly(d):
    return np.stack([d.values[d.time.dt.month.values == m].mean(0) for m in range(1, 13)])   # (12, lat, lon)

rows, CYCLE = [], {}
for var in ("tasmax", "pr"):
    e, h = monthly(load("era5", var)), monthly(load("gcm_historical", var))
    bias = np.where(h > 0.1, 100 * (e - h) / h, np.nan) if var == "pr" else e - h
    smooth = np.stack([np.nanmean(bias[[(m - 1) % 12, m, (m + 1) % 12]], axis=0) for m in range(12)])
    for lab, b in (("calendar month", bias), ("3-month window", smooth)):
        step = np.abs(np.roll(b, -1, axis=0) - b)                        # month m -> m+1 (Dec -> Jan included)
        rows.append({"variable": var, "bias": lab, "unit": "%" if var == "pr" else "degC",
                     "|bias change| between months: land mean": float(np.nanmean(rmean(step))),
                     "p95 over pixels": float(np.nanpercentile(np.where(MASK, step, np.nan), 95)),
                     "max": float(np.nanmax(np.where(MASK, step, np.nan)))})
    CYCLE[var] = (rmean(bias), rmean(smooth))
display(pd.DataFrame(rows).set_index(["variable", "bias"]).round(2))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, var in zip(axes, ("tasmax", "pr")):
    ax.plot(range(1, 13), CYCLE[var][0], "o-", label="calendar month"); ax.plot(range(1, 13), CYCLE[var][1], "s--", label="3-month window")
    ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"bias ERA5 - EC-Earth3 historical: {var} ({'%' if var == 'pr' else 'degC'}), land mean", fontsize=9)
    ax.set_xlabel("month")
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "bias_cycle.png", dpi=110); plt.show()

## 3. Extremes beyond the historical range

hq = 99.5th percentile of EC-Earth3 historical (all days, the 3 months centred on each month). For each run: share of
the wet days (>= 0.1 mm/day) above hq, and the excess x / hq of those days (median, 99th percentile, maximum, share
above x2 and x5). Plain multiplicative QDM transfers x / hq as a ratio onto ERA5; the rule of the F1 transfers the
excess as an amount instead (Section 5.1 of the method note).

In [ ]:
h = PR["gcm_historical"]
hmon = h.time.dt.month.values
HQ = np.stack([np.nanquantile(h.values[np.isin(hmon, bu._window(m, 3))], Q_TOP, axis=0) for m in range(1, 13)])
rows, SHARE = [], {}
for n in RUNS:
    if n not in PR:
        continue
    d = PR[n]; mon = d.time.dt.month.values
    above, wet, ratios = np.zeros(HQ.shape[1:]), np.zeros(HQ.shape[1:]), []
    for m in range(1, 13):
        x = d.values[mon == m]
        hq = HQ[m - 1]
        w = x >= bu.THRESH_PR
        a = x > hq
        above += a.sum(0); wet += w.sum(0)
        r = (x / hq)[a & MASK[None]]
        ratios.append(r[np.isfinite(r)])
    r = np.concatenate(ratios)
    SHARE[n] = np.where(MASK, 100 * above / np.maximum(wet, 1), np.nan)
    rows.append({"run": LABEL[n], "wet days above hq (%)": float(rmean(SHARE[n])),
                 "x / hq: median": float(np.median(r)), "p99": float(np.percentile(r, 99)), "max": float(r.max()),
                 "share of these days x/hq > 2 (%)": 100 * float((r > 2).mean()), "x/hq > 5 (%)": 100 * float((r > 5).mean()),
                 "hq < 1 mm/day (% of land pixel-months)": 100 * float(np.nanmean(np.where(MASK[None], HQ < 1, np.nan)))})
ext = pd.DataFrame(rows).set_index("run").round(3)
display(ext)
fig, axes = plt.subplots(1, len(SHARE), figsize=(4.5 * len(SHARE), 3.2), squeeze=False)
for ax, (n, s) in zip(axes[0], SHARE.items()):
    da = xr.DataArray(s, dims=("lat", "lon"), coords={"lat": h.lat, "lon": h.lon})
    im = ax.pcolormesh(da.lon, da.lat, da, shading="auto", cmap="magma_r", vmin=0, vmax=max(1.0, float(np.nanpercentile(s, 98))))
    plt.colorbar(im, ax=ax, shrink=0.8, label="%"); ax.set_facecolor("0.85"); ax.set_title(f"{LABEL[n]}: wet days above hq (%)", fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "days_above_hq.png", dpi=110); plt.show()

## How to read it

- **Dry days**: if EC-Earth3 and ERA5 have similar fractions at 0.1 mm, the dry-day treatment matters little; a large
  gap (drizzle) means it does real work, and its effect on the signal (WDF vs jitter) matters.
- **Bias cycle**: the "calendar month" row is roughly the size of the jumps of a monthly correction; the "3-month
  window" row what Cannon's windows would leave. If the jumps are already small (e.g. < 0.3 degC for tasmax), the
  windows are not needed.
- **Extremes**: by construction about 0.5 % of the historical days are above hq. If the runs have about 1-2 % of their
  wet days above it with x / hq mostly below 2, the rule above hq concerns few days and moderate excesses: its choice
  is secondary. Many days with x / hq > 5 (typically where hq < 1 mm/day) is where plain QDM blows up.